# CORAL: Merged Embedding Multi-Head (Clean Notebook)

This notebook rewrites the original research notebook into a clear, linear pipeline that is easy to run end-to-end in Colab. It covers data setup, model training, and retrieval evaluation, and saves checkpoints, metrics, and FAISS indexes.

## 0. Setup

In [ ]:
%pip -q install torch transformers sentence-transformers faiss-cpu \n
  langchain-community langchain-huggingface \n
  llama-index llama-index-retrievers-bm25 \n
  bertopic pytrec_eval PyStemmer tqdm scikit-learn pandas pynndescent

In [ ]:
from pathlib import Path
import os
import json
import pickle
import random
import collections
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 1212
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
DATA_ROOT = BASE_DIR / "data"
DATASET_DIR = DATA_ROOT / "raw" / "dataset"
FAISS_DIR = DATA_ROOT / "vector_store_doc"
BERTOPIC_DIR = DATA_ROOT / "bertopic"
PROCESSED_DIR = DATA_ROOT / "processed"
OUTPUT_DIR = BASE_DIR / "outputs" / "coral_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "history_num": 2,
    "batch_size": 64,
    "dropout": 0.25,
    "lr_stage1": 2e-4,
    "lr_stage2": 1e-3,
    "epochs_stage1": 26,
    "epochs_stage2": 16,
    "alpha_stage1": 0.9,
    "alpha_stage2": 0.8,
    "num_heads": 8,
    "n_layers": 2,
    "bidirectional": True,
    "quick_run": False,
    "quick_max_samples": 2000,
    "save_checkpoints": True,
    "save_metrics": True,
    "use_bertopic": False,
    "build_faiss": True,
    "bm25_top_k": 25,
    "eval_top_k": 20,
}

MODEL_SHORT = CONFIG["embedding_model"].split("/")[-1]
TRAIN_FILE = DATASET_DIR / "train" / "train_conversation.json"
TEST_FILE = DATASET_DIR / "test" / "test_conversation.json"
PASSAGE_FILE = DATASET_DIR / "passage_corpus.json"
QREL_TRAIN = DATASET_DIR / "train" / "train_qrel.tsv"
QREL_TEST = DATASET_DIR / "test" / "test_qrel.tsv"

VS_CONTEXT_PATH = FAISS_DIR / f"faiss_CORAL_{MODEL_SHORT}"
BERTOPIC_PATH = BERTOPIC_DIR / f"BerTopic_corpus_{MODEL_SHORT}"
PRE_TRAIN_PATH = PROCESSED_DIR / f"train_data_bertopic_{MODEL_SHORT}_negBM25.pkl"
PRE_TEST_PATH = PROCESSED_DIR / f"test_data_bertopic_{MODEL_SHORT}_negBM25.pkl"

print(f"Device: {device}")
print(f"BASE_DIR: {BASE_DIR}")
print(f"DATASET_DIR: {DATASET_DIR}")

## 1. Data Availability

In [ ]:
missing = [p for p in [TRAIN_FILE, TEST_FILE, PASSAGE_FILE] if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing dataset files. Please upload CORAL to /content/data/raw/dataset. "
        "Expected train/train_conversation.json, test/test_conversation.json, and passage_corpus.json."
    )

print(TRAIN_FILE)
print(TEST_FILE)
print(PASSAGE_FILE)

In [ ]:
from typing import List, Dict
from tqdm.auto import tqdm

import faiss
import Stemmer
from transformers import AutoTokenizer, AutoModel
from langchain_core.documents import Document
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings.huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores.faiss import DistanceStrategy
from llama_index.core.node_parser import SimpleFileNodeParser
from llama_index.retrievers.bm25 import BM25Retriever

try:
    from bertopic import BERTopic
except Exception:
    BERTopic = None


def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(
        input_mask_expanded.sum(1), min=1e-9
    )


def create_bm25_retriever(collection_path: Path, top_k: int = 25) -> BM25Retriever:
    docs = []
    with open(collection_path, "r") as handle:
        for line in tqdm(handle, desc="BM25 corpus"):
            passage = json.loads(line)
            docs.append(Document(metadata={"id_": passage["ref_id"]}, text=passage["ref_string"]))

    parser = SimpleFileNodeParser()
    md_nodes = parser.get_nodes_from_documents(docs)

    return BM25Retriever.from_defaults(
        nodes=md_nodes,
        similarity_top_k=top_k,
        stemmer=Stemmer.Stemmer("english"),
        language="english",
    )


def create_vector_store(doc_path: Path, model_name: str) -> FAISS:
    nlp_model = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": "cuda:0" if torch.cuda.is_available() else "cpu"},
        encode_kwargs={"normalize_embeddings": True, "batch_size": 256},
    )
    embed = AutoModel.from_pretrained(model_name)
    len_embed = embed.config.hidden_size

    index = faiss.IndexFlatIP(len_embed)
    vector_store = FAISS(
        embedding_function=nlp_model,
        index=index,
        docstore=InMemoryDocstore(),
        index_to_docstore_id={},
        distance_strategy=DistanceStrategy.COSINE,
    )

    tmp_doc = []
    tmp_pid = []
    with open(doc_path, "r") as f:
        for line in tqdm(f, desc="FAISS docs"):
            obj = json.loads(line)
            pid = obj["ref_id"]
            tmp_doc.append(Document(page_content=obj["ref_string"]))
            tmp_pid.append(pid)
            if len(tmp_pid) >= 256:
                vector_store.add_documents(documents=tmp_doc, ids=tmp_pid)
                tmp_doc, tmp_pid = [], []
    if tmp_pid:
        vector_store.add_documents(documents=tmp_doc, ids=tmp_pid)

    FAISS_DIR.mkdir(parents=True, exist_ok=True)
    vector_store.save_local(str(VS_CONTEXT_PATH))
    return vector_store


def maybe_build_bertopic(doc_path: Path, model_name: str) -> None:
    if not CONFIG["use_bertopic"]:
        return
    if BERTopic is None:
        raise RuntimeError("BERTopic is not available. Install bertopic or disable use_bertopic.")

    passages = []
    with open(doc_path, "r") as f:
        for line in tqdm(f, desc="BERTopic corpus"):
            passages.append(json.loads(line)["ref_string"])

    topic_model = BERTopic(embedding_model=model_name)
    topic_model.fit_transform(passages)
    BERTOPIC_DIR.mkdir(parents=True, exist_ok=True)
    topic_model.save(str(BERTOPIC_PATH))


class Attention(nn.Module):
    def __init__(self, hidden_size, batch_first=False):
        super().__init__()
        self.hidden_size = hidden_size
        self.batch_first = batch_first
        self.att_weights = nn.Parameter(torch.Tensor(1, hidden_size), requires_grad=True)
        stdv = 1.0 / np.sqrt(self.hidden_size)
        for weight in self.att_weights:
            nn.init.uniform_(weight, -stdv, stdv)

    def forward(self, inputs, lengths):
        if self.batch_first:
            batch_size, max_len = inputs.size()[:2]
        else:
            max_len, batch_size = inputs.size()[:2]

        weights = torch.bmm(
            inputs,
            self.att_weights.permute(1, 0).unsqueeze(0).repeat(batch_size, 1, 1),
        )
        attentions = torch.softmax(F.relu(weights.squeeze()), dim=-1)

        device = attentions.device
        if not isinstance(lengths, torch.Tensor):
            lengths = torch.tensor(lengths, device=device)
        else:
            lengths = lengths.to(device)
        idxs = torch.arange(max_len, device=device).unsqueeze(0)
        mask = (idxs < lengths.unsqueeze(1)).float()

        masked = attentions * mask
        sums = masked.sum(-1).unsqueeze(-1)
        attentions = masked / torch.clamp(sums, min=1e-9)

        weighted = inputs * attentions.unsqueeze(-1).expand_as(inputs)
        representations = weighted.sum(1).squeeze()
        return representations, attentions


class CTQE(nn.Module):
    def __init__(self, embed_size, dropout_rate, lstm_param, num_heads=8, run_cls=True):
        super().__init__()
        self.run_cls = run_cls
        self.norm1 = nn.LayerNorm(embed_size)
        self.dropout = nn.Dropout(dropout_rate)

        self.lstm_mem = nn.LSTM(
            embed_size,
            int(embed_size / 2),
            lstm_param["n_layers"],
            bidirectional=lstm_param["bidirectional"],
            dropout=dropout_rate,
            batch_first=True,
        )
        self.lstm_q = nn.LSTM(
            embed_size,
            int(embed_size / 2),
            lstm_param["n_layers"],
            bidirectional=lstm_param["bidirectional"],
            dropout=dropout_rate,
            batch_first=True,
        )

        self.weight_enc = nn.Linear(embed_size, 150) if lstm_param["bidirectional"] else nn.Linear(
            int(embed_size / 2), 150
        )
        self.cls_enc = nn.Linear(150, 1)
        self.mutihead_query = nn.MultiheadAttention(embed_size, num_heads, dropout=dropout_rate, batch_first=True)
        self.att_query = Attention(embed_size, batch_first=True)
        self.mutihead_ctx = nn.MultiheadAttention(embed_size, num_heads, dropout=dropout_rate, batch_first=True)

    def forward(self, mean_ids, ids, mean_ctx, ctx):
        mem_enc, _ = self.lstm_mem(ctx)
        q_enc_, _ = self.lstm_q(ids)
        q_enc = q_enc_.mean(dim=1, keepdim=True)

        p_enc = torch.einsum("bij,bkj->bki", q_enc, mem_enc).squeeze()
        weight_p = F.softmax(p_enc, dim=-1)
        h = torch.sum(weight_p.unsqueeze(-1) * mem_enc, dim=1)
        o = self.weight_enc(h + q_enc.squeeze())
        cls = torch.sigmoid(self.cls_enc(F.relu(o)))

        ctx_weight = torch.sum(weight_p.unsqueeze(-1) * ctx, dim=1).unsqueeze(1)
        ctx_att, _ = self.mutihead_ctx(q_enc, ctx, ctx)
        src, _ = self.mutihead_query(ctx_weight, q_enc_, ids)

        x = torch.cat([ids, self.dropout(src), self.dropout(ctx_att)], dim=1)
        lengths = torch.full((x.size(0),), x.size(1), dtype=torch.long, device=x.device)
        prediction, _ = self.att_query(x, lengths)

        prediction = cls * prediction + (1 - cls) * mean_ids if self.run_cls else prediction
        return prediction


class ContrastiveLoss(torch.nn.Module):
    def __init__(self, alpha=0.8):
        super().__init__()
        self.alpha = alpha

    def cos_sim(self, a, b):
        if not isinstance(a, torch.Tensor):
            a = torch.tensor(a)
        if not isinstance(b, torch.Tensor):
            b = torch.tensor(b)
        if len(a.shape) == 1:
            a = a.unsqueeze(0)
        if len(b.shape) == 1:
            b = b.unsqueeze(0)
        a_norm = torch.nn.functional.normalize(a, p=2, dim=1)
        b_norm = torch.nn.functional.normalize(b, p=2, dim=1)
        return torch.mm(a_norm, b_norm.transpose(0, 1))

    def forward(self, embeddings_src, embeddings_target, relevant_passage, irrelevant_passage):
        scores = self.cos_sim(embeddings_src, embeddings_target)
        re_pos = []
        re_neg = []
        for i in range(embeddings_src.size(0)):
            if relevant_passage[i] is not None:
                passage = relevant_passage[i].to(embeddings_src.device)
                pos_score = 1 + torch.sum(torch.exp(self.cos_sim(embeddings_src[i], passage)))
            else:
                pos_score = torch.tensor(1.0, device=embeddings_src.device)

            re_pos.append(pos_score)
            neg_score = torch.sum(torch.exp(self.cos_sim(embeddings_src[i], irrelevant_passage[i])))
            re_neg.append(pos_score + neg_score)

        loss_pos = torch.log(torch.stack(re_pos))
        loss_neg = torch.log(torch.stack(re_neg))
        loss_main = torch.log(torch.exp(torch.diagonal(scores, 0)))
        loss = torch.mean(-(self.alpha * loss_main + (1 - self.alpha) * (loss_pos - loss_neg)))
        return loss


def train(data_loader, model, criterion, optimizer, device):
    model.train()
    epoch_losses = []
    epoch_sim = []
    for batch in tqdm(data_loader, desc="training..."):
        mean_ids = batch["mean_query"].to(device)
        label = batch["rewrite"].to(device)
        ids = batch["embed_query"].to(device)
        mean_ctx = batch["mean_context"].to(device)
        ctx = batch["context"].to(device)
        batch_irrelevant = batch["batch_irrelevant"].to(device)

        output = model(mean_ids, ids, mean_ctx, ctx)
        loss = criterion(output, label, batch["batch_relevant"], batch_irrelevant)
        sim = F.cosine_similarity(output, label).detach().mean().item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())
        epoch_sim.append(sim)
    return float(np.mean(epoch_losses)), float(np.mean(epoch_sim))


def evaluate(data_loader, model, criterion, device):
    model.eval()
    epoch_losses = []
    epoch_sim = []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="evaluating..."):
            mean_ids = batch["mean_query"].to(device)
            label = batch["rewrite"].to(device)
            ids = batch["embed_query"].to(device)
            mean_ctx = batch["mean_context"].to(device)
            ctx = batch["context"].to(device)
            batch_irrelevant = batch["batch_irrelevant"].to(device)

            output = model(mean_ids, ids, mean_ctx, ctx)
            sim = F.cosine_similarity(output, label).detach().mean().item()
            loss = criterion(output, label, batch["batch_relevant"], batch_irrelevant)

            epoch_losses.append(loss.item())
            epoch_sim.append(sim)
    return float(np.mean(epoch_losses)), float(np.mean(epoch_sim))


def initialize_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.LSTM):
        for name, param in m.named_parameters():
            if "bias" in name:
                nn.init.zeros_(param)
            elif "weight" in name:
                nn.init.orthogonal_(param)


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


class EarlyStopper:
    def __init__(self, patience=1, min_delta=0, save_path="outputs"):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = float("inf")
        self.save_path = save_path

    def early_stop(self, validation_loss, model):
        os.makedirs(self.save_path, exist_ok=True)
        torch.save(model.state_dict(), os.path.join(self.save_path, "ctqe_last.pt"))
        if validation_loss < self.min_validation_loss:
            self.min_validation_loss = validation_loss
            self.counter = 0
            torch.save(model.state_dict(), os.path.join(self.save_path, "ctqe_best.pt"))
        elif validation_loss > (self.min_validation_loss + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                return True
        return False


class PreprocessedDataset:
    def __init__(self, filename, history_num, training=True, bm25_retriever=None, bertopic_model=None):
        self.filename = filename
        self.history_num = history_num
        self.training = training
        self.bm25_retriever = bm25_retriever
        self.bertopic_model = bertopic_model

        with open(filename, "r") as f:
            self.total_data = json.load(f)

    def __call__(self):
        res_ls = []
        for cnv in tqdm(self.total_data, desc=f"Preprocess {self.filename}"):
            cnv_name = cnv["conv_id"]
            turns = cnv["turns"]
            history_ls = []
            history_session = []

            for turn in turns:
                context = history_ls[-self.history_num :]
                if len(context) < self.history_num:
                    context = context + [""] * (self.history_num - len(context))

                nodes = self.bm25_retriever.retrieve(turn["question"]) if self.bm25_retriever else []
                irrelevant_docs_pids = [
                    str(node.metadata["id_"]) for node in nodes if str(node.metadata["id_"]) not in turn["golden_docs_pids"]
                ][:10]

                context_topics = None
                if self.bertopic_model is not None:
                    topics, _ = self.bertopic_model.transform(context)
                    context_topics = topics

                result_turn = {
                    "id": f"{cnv_name}_{turn['turn_id']}",
                    "question": turn["question"],
                    "rewrite": turn["golden_rewrite"] if self.training else turn.get("golden_rewrite"),
                    "relevant_docs_pids": turn["golden_docs_pids"],
                    "irrelevant_docs_pids": irrelevant_docs_pids,
                    "context": context,
                    "context_topics": context_topics,
                    "context_session": "[SEP]" + "[SEP]".join(history_session[-self.history_num :]),
                }

                res_ls.append(result_turn)
                history_ls.append(turn["question"] + " [SEP] " + turn["response"])
                history_session.append(turn["question"])

        return res_ls


class RetrieverDataset(torch.utils.data.Dataset):
    def __init__(self, filename, history_num=2, training=True, pre_data_path=None, bm25_retriever=None, bertopic_model=None):
        self.filename = filename
        self.history_num = history_num
        self.training = training
        self.bertopic_model = bertopic_model
        self.tokenizer = AutoTokenizer.from_pretrained(CONFIG["embedding_model"])
        self.embedding_model = AutoModel.from_pretrained(CONFIG["embedding_model"]).to(device)
        self.embedding_model.eval()

        if pre_data_path is None:
            pre = PreprocessedDataset(
                filename,
                history_num,
                training,
                bm25_retriever=bm25_retriever,
                bertopic_model=bertopic_model,
            )
            self.pre_data = pre()
        else:
            with open(pre_data_path, "rb") as handle:
                self.pre_data = pickle.load(handle)

    def __len__(self):
        return len(self.pre_data)

    def __getitem__(self, idx):
        entry = self.pre_data[idx]
        return {
            "id": entry["id"],
            "question": entry["question"],
            "rewrite": entry["rewrite"],
            "relevant_docs_pids": entry["relevant_docs_pids"],
            "irrelevant_docs_pids": entry["irrelevant_docs_pids"],
            "context": entry["context"],
            "context_topics": entry.get("context_topics"),
            "context_session": entry["context_session"],
        }


def get_collate_fn(vector_store, tokenizer, model_embedding, bertopic_model=None, use_bertopic=False):
    index_to_docstore_id = vector_store.index_to_docstore_id
    docstore_id_to_index = {y: x for x, y in index_to_docstore_id.items()}
    dim = vector_store.index.d

    def get_vectors(ids):
        if len(ids) == 0:
            return torch.zeros(1, dim)
        return torch.stack(
            [torch.tensor(vector_store.index.reconstruct_n(docstore_id_to_index[str(j)], 1)[0]) for j in ids],
            dim=0,
        )

    def collate_fn(batch):
        question_id = [i["id"] for i in batch]
        question_text = [i["question"] for i in batch]
        query_token_ids = tokenizer(question_text, padding=True, truncation=True, return_tensors="pt").to(device)

        relevant_docs_pids = [i["relevant_docs_pids"] for i in batch]
        irrelevant_docs_pids = [i["irrelevant_docs_pids"] for i in batch]

        batch_relevant = [get_vectors(i).to(device) if len(i) != 0 else None for i in relevant_docs_pids]
        batch_irrelevant = torch.stack([get_vectors(i) for i in irrelevant_docs_pids], dim=0).to(device)

        session = [i["context_session"] for i in batch]
        session_token_ids = tokenizer(session, padding=True, truncation=True, return_tensors="pt").to(device)

        with torch.no_grad():
            embed_query = model_embedding(**query_token_ids)
            embed_sess = model_embedding(**session_token_ids)

        mean_query = mean_pooling(embed_query, query_token_ids["attention_mask"])

        if use_bertopic and bertopic_model is not None:
            context_topics = [i["context_topics"] for i in batch]
            batch_ctx = torch.stack(
                [torch.tensor(bertopic_model.topic_embeddings_[i]) for i in context_topics],
                dim=0,
            ).to(device)
            context = torch.cat([batch_ctx, embed_sess[0]], dim=1)
            mean_context = batch_ctx.mean(dim=1)
        else:
            context = embed_sess[0]
            mean_context = mean_pooling(embed_sess, session_token_ids["attention_mask"])

        mean_rewrite = None
        if batch[0]["rewrite"] is not None:
            rewrite = [i["rewrite"] for i in batch]
            rewrite_token_ids = tokenizer(rewrite, padding=True, truncation=True, return_tensors="pt").to(device)
            with torch.no_grad():
                embed_rewrite = model_embedding(**rewrite_token_ids)
            mean_rewrite = mean_pooling(embed_rewrite, rewrite_token_ids["attention_mask"])

        return {
            "question_id": question_id,
            "question_text": question_text,
            "embed_query": embed_query[0],
            "mean_query": mean_query,
            "context": context,
            "mean_context": mean_context,
            "rewrite": mean_rewrite,
            "relevant_docs_pids": relevant_docs_pids,
            "batch_relevant": batch_relevant,
            "batch_irrelevant": batch_irrelevant,
        }

    return collate_fn

## 2. Build or Load FAISS Index and BERTopic

In [ ]:
if not VS_CONTEXT_PATH.exists():
    if not CONFIG["build_faiss"]:
        raise FileNotFoundError(f"Missing FAISS index at {VS_CONTEXT_PATH}")
    print("Building FAISS index... this can take a while.")
    vector_store = create_vector_store(PASSAGE_FILE, CONFIG["embedding_model"])
else:
    nlp_model = HuggingFaceEmbeddings(
        model_name=CONFIG["embedding_model"],
        model_kwargs={"device": "cuda:0" if torch.cuda.is_available() else "cpu"},
        encode_kwargs={"normalize_embeddings": True, "batch_size": 256},
    )
    vector_store = FAISS.load_local(
        str(VS_CONTEXT_PATH),
        nlp_model,
        allow_dangerous_deserialization=True,
        distance_strategy=DistanceStrategy.COSINE,
    )

bertopic_model = None
if CONFIG["use_bertopic"]:
    if BERTopic is None:
        raise RuntimeError("BERTopic is not available. Install bertopic or disable use_bertopic.")
    if not BERTOPIC_PATH.exists():
        print("Building BERTopic model... this can take a while.")
        maybe_build_bertopic(PASSAGE_FILE, CONFIG["embedding_model"])
    bertopic_model = BERTopic.load(str(BERTOPIC_PATH))

print(f"FAISS path: {VS_CONTEXT_PATH}")
print(f"BERTopic path: {BERTOPIC_PATH}")

## 3. Preprocess Dataset and Cache

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

bm25_retriever = create_bm25_retriever(PASSAGE_FILE, top_k=CONFIG["bm25_top_k"])

if not PRE_TRAIN_PATH.exists():
    train_data = RetrieverDataset(
        str(TRAIN_FILE),
        history_num=CONFIG["history_num"],
        training=True,
        pre_data_path=None,
        bm25_retriever=bm25_retriever,
        bertopic_model=bertopic_model,
    )
    with open(PRE_TRAIN_PATH, "wb") as f:
        pickle.dump(train_data.pre_data, f)

if not PRE_TEST_PATH.exists():
    test_data = RetrieverDataset(
        str(TEST_FILE),
        history_num=CONFIG["history_num"],
        training=False,
        pre_data_path=None,
        bm25_retriever=bm25_retriever,
        bertopic_model=bertopic_model,
    )
    with open(PRE_TEST_PATH, "wb") as f:
        pickle.dump(test_data.pre_data, f)

print(PRE_TRAIN_PATH)
print(PRE_TEST_PATH)

## 4. DataLoaders

In [ ]:
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

train_data = RetrieverDataset(
    str(TRAIN_FILE),
    history_num=CONFIG["history_num"],
    training=True,
    pre_data_path=str(PRE_TRAIN_PATH),
    bertopic_model=bertopic_model,
)
test_data = RetrieverDataset(
    str(TEST_FILE),
    history_num=CONFIG["history_num"],
    training=False,
    pre_data_path=str(PRE_TEST_PATH),
    bertopic_model=bertopic_model,
)

base_train_data = train_data

if CONFIG["quick_run"]:
    max_samples = min(CONFIG["quick_max_samples"], len(base_train_data))
    indices = torch.arange(max_samples)
    train_subset = torch.utils.data.Subset(base_train_data, indices)
else:
    train_subset = base_train_data

train_size = int(0.7 * len(train_subset))
valid_size = len(train_subset) - train_size
generator = torch.Generator().manual_seed(SEED)
train_dataset, valid_dataset = torch.utils.data.random_split(
    train_subset, [train_size, valid_size], generator=generator
)

collate_fn = get_collate_fn(
    vector_store,
    tokenizer=base_train_data.tokenizer,
    model_embedding=base_train_data.embedding_model,
    bertopic_model=bertopic_model,
    use_bertopic=CONFIG["use_bertopic"],
)

train_loader = DataLoader(
    train_dataset,
    sampler=RandomSampler(train_dataset),
    batch_size=CONFIG["batch_size"],
    collate_fn=collate_fn,
)
valid_loader = DataLoader(
    valid_dataset,
    sampler=SequentialSampler(valid_dataset),
    batch_size=CONFIG["batch_size"],
    collate_fn=collate_fn,
)
test_loader = DataLoader(
    test_data,
    sampler=SequentialSampler(test_data),
    batch_size=CONFIG["batch_size"],
    collate_fn=collate_fn,
)

print(f"Train size: {len(train_dataset)}")
print(f"Valid size: {len(valid_dataset)}")
print(f"Test size: {len(test_data)}")

## 5. Model

In [ ]:
lstm_param = {
    "n_layers": CONFIG["n_layers"],
    "bidirectional": CONFIG["bidirectional"],
}

model = CTQE(
    train_data.embedding_model.config.hidden_size,
    CONFIG["dropout"],
    lstm_param,
    CONFIG["num_heads"],
)
model.apply(initialize_weights)
model = model.to(device)

print(f"Trainable parameters: {count_parameters(model):,}")

## 6. Training

In [ ]:
import torch.optim as optim

def run_training_stage(stage_name, lr, alpha, epochs, patience=3):
    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=1e-6,
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=4 if stage_name == "stage1" else 10, gamma=1
    )
    criterion = ContrastiveLoss(alpha=alpha).to(device)
    early_stopper = EarlyStopper(patience=patience, min_delta=1e-4, save_path=str(OUTPUT_DIR))
    metrics = collections.defaultdict(list)

    for epoch in range(epochs):
        train_loss, train_sim = train(train_loader, model, criterion, optimizer, device)
        scheduler.step()
        valid_loss, valid_sim = evaluate(valid_loader, model, criterion, device)
        metrics["train_losses"].append(train_loss)
        metrics["train_sim"].append(train_sim)
        metrics["valid_losses"].append(valid_loss)
        metrics["valid_sim"].append(valid_sim)
        print(f"{stage_name} epoch {epoch}: train_loss={train_loss:.5f} valid_loss={valid_loss:.5f}")

        if early_stopper.early_stop(valid_loss, model):
            break

    if CONFIG["save_checkpoints"]:
        torch.save(model.state_dict(), OUTPUT_DIR / f"{stage_name}_final.pt")

    return metrics

metrics_stage1 = run_training_stage(
    stage_name="stage1",
    lr=CONFIG["lr_stage1"],
    alpha=CONFIG["alpha_stage1"],
    epochs=CONFIG["epochs_stage1"],
)

In [ ]:
metrics_stage2 = run_training_stage(
    stage_name="stage2",
    lr=CONFIG["lr_stage2"],
    alpha=CONFIG["alpha_stage2"],
    epochs=CONFIG["epochs_stage2"],
)

## 7. Evaluation (FAISS Retrieval + pytrec_eval)

In [ ]:
import pytrec_eval

def evaluate_run(res_model, res_gold):
    metrics = [
        "recall_5",
        "recall_10",
        "recall_20",
        "ndcg_cut_3",
        "map",
        "map_cut_10",
        "recip_rank",
    ]
    evaluator = pytrec_eval.RelevanceEvaluator(res_gold, set(metrics))
    out = {m: [] for m in metrics}
    for _, scores in evaluator.evaluate(res_model).items():
        for m in metrics:
            out[m].append(scores[m])
    return {m: float(np.mean(out[m])) for m in metrics}


def evaluate_by_vector(embeddings, batch_ids, batch_gold, top_k):
    res_model = {}
    res_gold = {}
    for i, qid in enumerate(batch_ids):
        gold = {str(pid): 1 for pid in batch_gold[i]}
        nodes = vector_store.similarity_search_with_score_by_vector(
            embedding=embeddings[i].tolist(),
            k=top_k,
        )
        model_retrieve = {node[0].id: float(node[1]) for node in nodes}
        res_model[qid] = model_retrieve
        res_gold[qid] = gold
    return res_model, res_gold


def evaluate_ctqe(model, loader, top_k):
    model.eval()
    res_model = {}
    res_gold = {}
    time_inference = 0.0
    count = 0

    with torch.no_grad():
        for batch in loader:
            mean_ids = batch["mean_query"].to(device)
            ids = batch["embed_query"].to(device)
            mean_ctx = batch["mean_context"].to(device)
            ctx = batch["context"].to(device)
            question_id = batch["question_id"]
            relevant_docs_pids = batch["relevant_docs_pids"]

            start = time.time()
            output = model(mean_ids, ids, mean_ctx, ctx)
            embed_question = F.normalize(output, p=2, dim=1)
            time_inference += time.time() - start
            count += len(question_id)

            for i, qid in enumerate(question_id):
                gold = {str(pid): 1 for pid in relevant_docs_pids[i]}
                nodes = vector_store.similarity_search_with_score_by_vector(
                    embedding=embed_question[i].tolist(),
                    k=top_k,
                )
                model_retrieve = {node[0].id: float(node[1]) for node in nodes}
                res_gold[qid] = gold
                res_model[qid] = model_retrieve

    results = evaluate_run(res_model, res_gold)
    results["time_inference_total"] = time_inference
    results["time_inference_per_query"] = time_inference / max(count, 1)
    return results


def evaluate_dense_baseline(loader, top_k):
    res_model = {}
    res_gold = {}
    for batch in loader:
        embeddings = F.normalize(batch["mean_query"], p=2, dim=1)
        question_id = batch["question_id"]
        relevant_docs_pids = batch["relevant_docs_pids"]
        batch_res_model, batch_res_gold = evaluate_by_vector(
            embeddings, question_id, relevant_docs_pids, top_k
        )
        res_model.update(batch_res_model)
        res_gold.update(batch_res_gold)
    return evaluate_run(res_model, res_gold)


def evaluate_rewrite_baseline(loader, top_k):
    res_model = {}
    res_gold = {}
    for batch in loader:
        embeddings = batch["rewrite"]
        if embeddings is None:
            continue
        embeddings = F.normalize(embeddings, p=2, dim=1)
        question_id = batch["question_id"]
        relevant_docs_pids = batch["relevant_docs_pids"]
        batch_res_model, batch_res_gold = evaluate_by_vector(
            embeddings, question_id, relevant_docs_pids, top_k
        )
        res_model.update(batch_res_model)
        res_gold.update(batch_res_gold)
    return evaluate_run(res_model, res_gold)


def evaluate_bm25(loader, top_k):
    res_model = {}
    res_gold = {}
    for batch in loader:
        question_id = batch["question_id"]
        question_text = batch["question_text"]
        relevant_docs_pids = batch["relevant_docs_pids"]
        for i, qid in enumerate(question_id):
            nodes = bm25_retriever.retrieve(question_text[i])
            model_retrieve = {str(node.metadata["id_"]): float(node.score) for node in nodes[:top_k]}
            res_model[qid] = model_retrieve
            res_gold[qid] = {str(pid): 1 for pid in relevant_docs_pids[i]}
    return evaluate_run(res_model, res_gold)


top_k = CONFIG["eval_top_k"]

results_ctqe = evaluate_ctqe(model, test_loader, top_k)
results_dense = evaluate_dense_baseline(test_loader, top_k)
results_rewrite = evaluate_rewrite_baseline(test_loader, top_k)
results_bm25 = evaluate_bm25(test_loader, top_k)

all_results = {
    "ctqe": results_ctqe,
    "dense_baseline": results_dense,
    "rewrite_baseline": results_rewrite,
    "bm25_baseline": results_bm25,
}

all_results

## 8. Save Outputs

In [ ]:
if CONFIG["save_metrics"]:
    metrics_path = OUTPUT_DIR / "retrieval_metrics.json"
    with open(metrics_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"Saved metrics: {metrics_path}")

if CONFIG["save_checkpoints"]:
    print(f"Checkpoints directory: {OUTPUT_DIR}")

print(f"FAISS index: {VS_CONTEXT_PATH}")
print(f"BERTopic model: {BERTOPIC_PATH}")